# RetroTAC quickstart — score PROTAC synthesizability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ribesstefano/RetroTAC/blob/main/notebooks/retrotac_quickstart.ipynb)

[RetroTAC](https://github.com/ribesstefano/RetroTAC) predicts how synthesizable a PROTAC is
directly from its SMILES string, without calling a retrosynthesis planner at inference time.
The published model is a Caruana-weighted ensemble of 27 cross-validation members (13 GNN,
7 MLP, 7 XGBoost) that reports the spread across its members as an uncertainty estimate.

This notebook:

1. installs `retrotac`,
2. downloads the paper's held-out test set (`data/sets/routes_test.csv`),
3. loads the ensemble from the Hugging Face Hub ([`ailab-bio/RetroTAC`](https://huggingface.co/ailab-bio/RetroTAC)),
4. scores a sample of held-out molecules and compares the predictions against the
   route-derived ground truth,
5. scores your own SMILES.

A CPU runtime is enough — scoring the 300 molecules below takes about a minute once the model
is downloaded. Select a GPU runtime (*Runtime → Change runtime type → T4 GPU*) to score a
larger library.

## 1. Install

In [ ]:
# ~2-4 min: pip clones the repository and installs RDKit, PyTorch and Chemprop.
%pip install -q "retrotac @ git+https://github.com/ribesstefano/RetroTAC.git"

> **If Colab asks you to restart the runtime**, do it (*Runtime → Restart session*), then carry
> on from the next cell — you do not need to re-run the install.

RetroTAC targets **Python 3.12**, which is what Colab ships. The check below is only there to
give a clear message if you are running this elsewhere on a different interpreter.

> [!IMPORTANT]
> **Running locally on macOS?** The next cell pins `OMP_NUM_THREADS=1` before importing torch,
> which is required there: `torch` and `scikit-learn` each bundle their own `libomp`, and with
> both OpenMP runtimes multi-threaded in one process the kernel **segfaults part-way through the
> ensemble** (typically around member 3 of 27). Run it as the first cell of a fresh kernel — if
> this kernel has already imported torch, restart it (*Kernel → Restart*) and start from here.
> On Colab and Linux the cell is a no-op.

In [ ]:
import os
import platform
import sys

# macOS only: torch and scikit-learn each ship their own copy of libomp, and
# loading both OpenMP runtimes into one process segfaults the kernel part-way
# through the ensemble (the GNN members are torch, the XGB/MLP preprocessors are
# scikit-learn). Pinning a single OpenMP thread avoids it.
#
# This MUST run before torch/sklearn are first imported, so keep it in the cell
# above the imports. If your kernel has already imported them, restart it
# (*Kernel -> Restart*) and run this cell first.
if platform.system() == "Darwin":
    os.environ.setdefault("OMP_NUM_THREADS", "1")

print("Python:", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    print("WARNING: RetroTAC pins Python 3.12 (`requires-python = \"==3.12.*\"`); "
          "the install above may have failed on this interpreter.")

import numpy as np
import pandas as pd
import torch

from retrotac.models.ensemble import RetroTAC
from retrotac.models.metrics import compute_all_metrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| OMP_NUM_THREADS:", os.environ.get("OMP_NUM_THREADS", "unset"))

## 2. Get the held-out test set

`data/sets/routes_test.csv` holds the molecules the paper reports its held-out metrics on,
together with their route-derived label. It is not checked into git — it ships inside
`release/retrotac_data.tar.gz`, which also carries every other pipeline input and intermediate
(see [Reproducibility](https://github.com/ribesstefano/RetroTAC#-reproducibility)).

Two columns matter here:

| Column | Meaning |
|---|---|
| `smiles` | the PROTAC |
| `synthesizability` | the route-derived score in [0, 1] that RetroTAC is trained to predict — higher is easier to make |

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/ribesstefano/RetroTAC/main/release/retrotac_data.tar.gz"
TEST_CSV = "data/sets/routes_test.csv"

archive = Path("retrotac_data.tar.gz")
workdir = Path("retrotac_data")

if not archive.exists():
    print(f"Downloading {DATA_URL} (~19 MB) ...")
    urllib.request.urlretrieve(DATA_URL, archive)

with tarfile.open(archive) as tar:
    tar.extract(TEST_CSV, path=workdir, filter="data")

df = pd.read_csv(workdir / TEST_CSV)
print(f"{len(df)} held-out molecules")
df[["smiles", "synthesizability"]].head()

Scoring all 2,232 molecules works fine, but a random sample keeps this notebook quick. Raise
`N_MOLECULES` to use more of the test set.

In [ ]:
N_MOLECULES, SEED = 300, 42

sample = df.sample(n=N_MOLECULES, random_state=SEED).reset_index(drop=True)
smiles = sample["smiles"].tolist()
y_true = sample["synthesizability"].to_numpy(dtype=float)

print(f"{len(smiles)} molecules sampled")
print(f"label: mean {y_true.mean():.3f}, min {y_true.min():.3f}, max {y_true.max():.3f}")

## 3. Load the ensemble from the Hub

`from_pretrained` takes a Hub repo id or a local directory. The first call downloads the 27
members (**~1.4 GB**, dominated by the 13 GNN checkpoints) into the Hugging Face cache; later
calls reuse it.

`load_strategy` controls how many members are held in memory as loaded model objects, not what
is downloaded:

- `"eager"` (default) — load all 27 up front and keep them resident. Fastest across repeated
  `predict` calls, heaviest on RAM.
- `"lazy"` with `max_resident` — load on first use and cache at most that many. Used below to
  stay comfortable inside a free Colab runtime.
- `"stream"` — load, predict, discard, one member at a time. Lowest memory, slowest.

In [ ]:
model = RetroTAC.from_pretrained(
    "ailab-bio/RetroTAC",
    device=DEVICE,
    load_strategy="lazy",
    max_resident=4,
)
print(f"Ensemble ready on {DEVICE}")

## 4. Score the molecules

`predict` featurizes the SMILES once and fans the features out to every member, then returns
the Caruana-weighted mean. With `return_details=True` you also get `std` — the spread across
the distinct member models, RetroTAC's uncertainty estimate — and every member's raw
prediction under `members`.

For a larger library, add `n_jobs=-1` to parallelize featurization across cores.

In [ ]:
out = model.predict(
    smiles,
    return_details=True,
    gnn_batch_size=256 if DEVICE == "cuda" else 64,  # raise on GPU, watching GPU memory
)

results = pd.DataFrame(
    {
        "smiles": smiles,
        "true": y_true,
        "predicted": out["mean"],
        "uncertainty": out["std"],
    }
)
print(f"{len(out['members'])} members contributed to each prediction")
results.head(10)

## 5. How good are the predictions?

`compute_all_metrics` returns the same regression and thresholded-classification metrics the
paper reports. On the **full** held-out set the ensemble reaches **RMSE 0.132** and
**R² 0.647**. A 300-molecule sample carries real sampling noise, so expect numbers in that
neighbourhood but not a match — they can land on either side (with `SEED = 42` this sample
scores RMSE ≈ 0.106, R² ≈ 0.78). Set `N_MOLECULES = len(df)` above to reproduce the paper's
figures exactly.

In [ ]:
metrics = compute_all_metrics(y_true, out["mean"])

for key in ["rmse", "mae", "r2", "pearson_r", "spearman_rho", "clf_roc_auc"]:
    print(f"{key:>14}: {metrics[key]:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.errorbar(
    results["true"], results["predicted"], yerr=results["uncertainty"],
    fmt="o", ms=4, alpha=0.55, lw=0.8, capsize=0,
)
lims = (-0.05, 1.05)
ax.plot(lims, lims, "k--", lw=1, label="perfect prediction")
ax.set(xlim=lims, ylim=lims,
       xlabel="route-derived synthesizability (true)",
       ylabel="RetroTAC prediction")
ax.set_title(f"n = {len(results)}   RMSE = {metrics['rmse']:.3f}   $R^2$ = {metrics['r2']:.3f}")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
plt.show()

Error bars are the per-molecule spread across ensemble members. They widen on molecules that
sit further from the training distribution, which makes them a useful filter when screening:
rank by the mean, then discount the predictions whose members disagree.

## 6. Score your own molecules

Pass any list of SMILES. The first two below are PROTACs from the held-out set. Aspirin is
there as an **out-of-distribution check**: RetroTAC is trained on PROTACs only, so a small
drug-like molecule is nothing like what it has seen.

In [ ]:
my_smiles = [
    # PROTAC: thalidomide-based CRBN ligand + PEG linker + warhead
    "COc1ccc(-c2ocnc2C(=O)NCCCc2cn(CCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)nn2)cc1I",
    # PROTAC: a second, differently linked degrader
    "C=C1C(=O)N(C2CCC(=O)NC2=O)C(=O)/C1=C/C(=C)N(C)CCC(C)N(C)CCC(C)C#Cc1ccc(Cn2nc(-c3ccc(C#N)c(Cl)c3)c(C)c2C)cc1",
    # aspirin -- deliberately out of distribution
    "CC(=O)Oc1ccccc1C(=O)O",
]

mine = model.predict(my_smiles, return_details=True)
pd.DataFrame(
    {
        "molecule": ["PROTAC 1", "PROTAC 2", "aspirin"],
        "smiles": [s[:40] + "..." if len(s) > 40 else s for s in my_smiles],
        "synthesizability": mine["mean"].round(3),
        "uncertainty": mine["std"].round(3),
    }
)

Aspirin lands mid-scale (≈ 0.44) with by far the widest uncertainty of the three (≈ 0.23,
against ≈ 0.08 for the first PROTAC) — the members disagree sharply because the molecule is
outside the training distribution. **That is the result to take away, not the score**: aspirin
is trivially synthesizable, and RetroTAC says nothing useful about it. Use the model inside the
chemical space it was fitted on, and let the spread flag the molecules where it is guessing.

## How to read the scores

- **Higher is easier to make.** The label rewards short retrosynthetic routes and the balanced
  joining of fragments, computed from routes found by the ShallowTree planner.
- **Predictions are not clipped to [0, 1].** A value outside that range is a signal rather than
  a bug: it means the molecule differs substantially from the training distribution.
- **The endpoints are not symmetric.** A molecule already in the purchasable stock scores 1,
  whereas a search that failed to reach stock within the planner's budget scores 0 — which may
  record genuine difficulty, or merely a timeout or depth limit.
- **Treat the continuous score and its ranking as the primary output.** The 0.7 threshold behind
  the `clf_*` metrics is a reporting convention, not a calibrated decision boundary.

## Where to go next

- [README](https://github.com/ribesstefano/RetroTAC#readme) — installation profiles and the
  single-model (GNN-only) quickstart
- [docs/README.md](https://github.com/ribesstefano/RetroTAC/blob/main/docs/README.md) — the
  reproducibility guide: route and molecule scoring, leakage controls, training, every figure
- [CONTRIBUTING.md](https://github.com/ribesstefano/RetroTAC/blob/main/CONTRIBUTING.md) —
  environments and the cross-validation scheme